# 07. risk: 시나리오를 여러 번 돌려 민감도·손익 표로

`src/quanttool/analysis/risk.py`

| 의존하는 것 | 제공하는 것 |
|---|---|
| `config` (`RiskConfig`), `market` (scenario, roll), `portfolio` (npvs), `scenarios` (bucket) | 민감도·손익 DataFrame |

**객체 지도** (이 순서로 본다)

1. 흐름: 생성 → 적용 → 재평가 → risk
2. `scenario_pnl`
3. `rule`
4. `key_rate`: delta·gamma, overrides, shift_tenors, 할인·예측 분리
5. `factor_delta`
6. `cross_gamma`
7. `theta`: ForwardForward, 평가일만 옮긴 경우 vs roll

> 데이터는 `tests/fixtures/`의 **합성 샘플**이다. 구조를 보는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIX = ROOT / "tests" / "fixtures"  # 합성 샘플 시세·예시 북
ASOF = date(2026, 10, 23)

from quanttool.config import load_config
from quanttool.market.quotes import read_quotes

cfg = load_config(ROOT / "configs")
quotes = read_quotes(FIX / "market_sample.txt")
q = quotes[ASOF]

from quanttool.analysis.risk import cross_gamma, factor_delta, key_rate, rule, scenario_pnl, theta
from quanttool.analysis.scenarios import from_loadings, parallel
from quanttool.config import Override
from quanttool.market import build_market
from quanttool.portfolio import Portfolio

market = build_market(ASOF, q, cfg)
book = Portfolio.from_file(FIX / "ust_book.yaml", products=cfg.products).build(market, cfg)
swaps = Portfolio.from_file(FIX / "krw_swaps.yaml", products=cfg.products).build(market, cfg)

## 흐름
```
scenarios: {요인: 이동} → market.scenario(s) → book.npvs() → risk: 반복·차분·환산·표
```
시나리오 하나는 risk 없이 세 줄. risk는 "어떻게 흔들어 몇으로 부를지"(그릭 정의)를 담는다.

In [ ]:
base = book.npvs()
with market.scenario(parallel(market, "UST", 0.001)):
    by_hand = book.npvs() - base
pd.DataFrame(
    {
        "손으로": by_hand,
        "scenario_pnl": scenario_pnl(book, market, {"p": parallel(market, "UST", 0.001)})
        .loc["p"]
        .drop("total"),
    }
).round(2)

## `scenario_pnl(book, market, {이름: 시나리오})` → 행: 시나리오, 열: 거래 + total

In [ ]:
obj = market.objects["UST"]
loading = {t: -1 + 2 * time / max(obj.times.values()) for t, time in obj.times.items()}
scenario_pnl(
    book,
    market,
    {
        "parallel +10bp": parallel(market, "UST", 0.001),
        "steepener 5bp": from_loadings("UST", loading, 0.0005),
    },
).round(2)

## `rule(거래, risk)`: 거래 id → 상품 종류 → 기본값 순으로 (이동 크기, 방식)

In [ ]:
risk_fd = cfg.risk.model_copy(
    update={"overrides": {"Bond": Override(shift_size=0.001, scheme="Central")}}
)
{i: (rule(t, cfg.risk), rule(t, risk_fd)) for i, t in book.trades.items()}

## `key_rate(book, market, risk, 커브)` → {"delta", "gamma"}: 만기 × 거래 (1bp당)
격자 만기마다 bucket을 올리고 내려 NPV 차이. 합 ≈ 평행 1bp (차이는 2차 항).

In [ ]:
kr = key_rate(book, market, cfg.risk, "UST")
p1 = scenario_pnl(book, market, {"p": parallel(market, "UST", 1e-4)}).loc["p", "total"]
print("Σ delta", round(kr["delta"]["total"].sum(), 2), "/ 평행 +1bp", round(p1, 2))
kr["delta"].round(1).head(10)

상품별 규칙(10bp 중심 차분 → 1bp 환산)과 성긴 이동 만기(`shift_tenors`).

In [ ]:
fd = key_rate(book, market, risk_fd, "UST")["delta"]
display(
    pd.DataFrame({"1bp Forward": kr["delta"]["UST_10Y"], "10bp Central → 1bp": fd["UST_10Y"]})
    .loc[["5Y", "10Y"]]
    .round(2)
)
coarse = cfg.risk.model_copy(update={"shift_tenors": {"UST": ["2Y", "5Y", "10Y", "30Y"]}})
key_rate(book, market, coarse, "UST")["delta"].round(1)

듀얼 커브 스왑: 예측(CD91)과 할인(KOFR) 민감도가 따로 나온다.

In [ ]:
kr2 = pd.concat(
    {c: key_rate(swaps, market, cfg.risk, c)["delta"] for c in ["CD91", "KOFR"]}, axis=1
)
kr2.reindex(market.objects["KOFR"].tenors).fillna(0).round(0)

## `factor_delta(book, market, {이름: 요인 키 또는 h → 시나리오}, size)`
아무 요인·규칙의 중심 차분 delta·gamma (size 이동당). 그릭 정의를 바꿔 보는 자리.

In [ ]:
factor_delta(
    book, market, {"UST/10Y": "UST/10Y", "parallel": lambda h: parallel(market, "UST", h)}, 1e-4
)["delta"].round(2)

## `cross_gamma(book, market, risk)`: 두 요인을 함께 움직일 때만 생기는 손익
이웃 격자점(9Y·10Y)은 같은 현금흐름에 걸려 0이 아니고, 떨어진 점(5Y·10Y)은 0.

In [ ]:
cg = cfg.risk.model_copy(update={"cross_gamma": [("UST/9Y", "UST/10Y"), ("UST/5Y", "UST/10Y")]})
cross_gamma(book, market, cg)

## `theta(book, market, d)`: roll(d) → 재평가 → roll(asof)
ForwardForward(오늘 커브의 d 시점 선도가 d의 spot)라서 theta ≈ 단기금리 carry이고 roll-down은 빠진다.

In [ ]:
th = theta(book, market, date(2026, 10, 26))
carry = book.npvs() * (1 / market["UST"].discount(3 / 365) - 1)
pd.DataFrame({"theta": th, "NPV × 3일 단기 carry": carry}).round(2)

평가일만 옮기면(커브는 10/23 그대로) 결제일·경과이자는 바뀌는데 NPV는 커브 기준일로 할인돼 날짜가 섞인다. `roll`은 시장도 옮긴다.

In [ ]:
from quanttool.market.curves import evaluation_date

bond = book.instruments["UST_10Y"]
d = date(2027, 2, 16)


def row(label):
    return {
        "경우": label,
        "결제일": bond.settlementDate().ISO(),
        "경과이자": bond.accruedAmount(),
        "clean": bond.cleanPrice(),
        "NPV": bond.NPV(),
    }


rows = []
with market.active():
    rows.append(row("10/23 시장"))
with evaluation_date(d):
    rows.append(row("평가일만 2/16"))
market.roll(d)
with market.active():
    rows.append(row("roll(2/16)"))
market.roll(ASOF)
pd.DataFrame(rows).set_index("경우").round(4)

## 요약

| 객체 | 입력 | 출력 | 언제 쓰나 |
|---|---|---|---|
| `scenario_pnl` | 북, 시장, {이름: 시나리오} | 시나리오 × 거래 | 스트레스·시나리오 손익 |
| `key_rate` | 북, 시장, `RiskConfig`, 커브 | {delta, gamma}: 만기 × 거래 | 만기별 금리 민감도 |
| `factor_delta` | 북, 시장, {이름: 요인/규칙}, 크기 | {delta, gamma} | 임의 요인·수정 그릭 |
| `cross_gamma` | 북, 시장, `RiskConfig` | 요인 쌍 × 거래 | 교차 민감도 |
| `theta` | 북, 시장, 다음 날 | 거래별 Series | 시간 경과 손익 |

## 내 메모

- 